In [ ]:
# ============================================================
# CONFIGURAÇÃO DE DIRETÓRIOS DO PROJETO
# ============================================================
#
# Esta célula localiza automaticamente a raiz do projeto a partir
# do arquivo .env e define o diretório usado para armazenar dados
# externos baixados ou recebidos pela pipeline.
#
# A ideia é permitir que cada integrante execute o notebook no seu
# próprio ambiente, sem alterar caminhos fixos no código-fonte.
#
# Estrutura esperada:
#
#   projeto/
#   ├── .env
#   ├── notebooks/
#   ├── src/
#   └── data_lake/
#       └── external/
#
# Ponto de configuração:
#
#   O arquivo .env deve existir na raiz do projeto.
#
# ============================================================
# BIBLIOTECAS E CLASSES
# ============================================================

from pathlib import Path
from dotenv import find_dotenv

# ============================================================
# DEFINIÇÃO DA RAIZ DO PROJETO
# ============================================================

# Localiza o arquivo .env a partir do diretório atual de execução.
# Isso evita depender de caminhos absolutos como C:/Users/...,
# que mudam entre os integrantes do projeto.
PROJECT_ROOT = Path(find_dotenv(usecwd=True)).parent

# Define o diretório da camada external dentro do data lake local.
# Essa camada representa arquivos externos ainda sem tratamento.
EXTERNAL_DIR = PROJECT_ROOT / "data_lake" / "external"

In [ ]:
# ============================================================
# CONFIGURAÇÃO DAS FONTES DE DADOS EXTERNAS
# ============================================================
#
# Esta célula define as URLs oficiais dos microdados do INEP que
# serão utilizados na etapa de aquisição dos dados.
#
# Os arquivos ainda não são baixados nesta etapa. Apenas registramos
# quais anos estão disponíveis para download.
#
# Ponto de configuração:
#
#   Para incluir um novo ano, adicione uma nova entrada no dicionário:
#
#   ANO: "URL_DO_ARQUIVO_ZIP"
#
# Exemplo:
#
#   2026: "https://download.inep.gov.br/..."
#
# ============================================================
# URLS DOS MICRODADOS
# ============================================================

MICRODADOS_INEP = {
    2023: "https://download.inep.gov.br/dados_abertos/microdados_avaliacao_da_alfabetizacao_2023.zip",
    2024: "https://download.inep.gov.br/dados_abertos/microdados_avaliacao_da_alfabetizacao_2024.zip",
    2025: "https://download.inep.gov.br/dados_abertos/microdados_AEEB_2025.zip",
}

# ============================================================
# CONFERÊNCIA DA CONFIGURAÇÃO
# ============================================================

# Exibe o diretório onde os arquivos serão salvos nas próximas etapas.
print(f"Destino dos downloads : {EXTERNAL_DIR}")

# Exibe os anos configurados para aquisição.
print(f"Anos disponíveis      : {list(MICRODADOS_INEP.keys())}")

In [ ]:
# ============================================================
# FUNÇÃO DE DOWNLOAD DOS ARQUIVOS EXTERNOS
# ============================================================
#
# Esta célula prepara a função responsável por baixar os arquivos
# ZIP configurados anteriormente no dicionário MICRODADOS_INEP.
#
# O download é feito em streaming, ou seja, o arquivo é salvo em
# partes no disco. Isso evita carregar arquivos grandes inteiros
# na memória.
#
# Observação:
#
#   O parâmetro verify=False desativa a validação do certificado SSL.
#   Ele deve ser usado apenas quando o ambiente apresentar problemas
#   de certificado ao acessar a fonte oficial.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

import requests
import urllib3

# ============================================================
# CONFIGURAÇÃO DA REQUISIÇÃO HTTP
# ============================================================

# Desativa avisos gerados pelo uso de verify=False nas requisições.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Define um User-Agent para reduzir a chance de bloqueios por requisições
# identificadas como automáticas pelo servidor.
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# ============================================================
# DOWNLOAD EM STREAMING
# ============================================================

def baixar_arquivo(url: str, destino: Path) -> Path:
    """
    Baixa um arquivo de uma URL para um caminho local.

    Parâmetros:
        url: endereço do arquivo que será baixado.
        destino: caminho local onde o arquivo será salvo.

    Retorno:
        Caminho local do arquivo baixado.
    """

    print(f"Baixando arquivo: {destino.name}")

    with requests.get(
        url,
        headers=HEADERS,
        stream=True,
        timeout=30,
        verify=False
    ) as resposta:
        # Interrompe a execução caso a URL retorne erro HTTP.
        resposta.raise_for_status()

        # Salva o arquivo em partes para evitar alto consumo de memória.
        with open(destino, "wb") as arquivo:
            for pedaco in resposta.iter_content(chunk_size=8 * 1024 * 1024):
                arquivo.write(pedaco)

    print(f"Download concluído: {destino.name}")

    return destino

In [ ]:
# ============================================================
# EXECUÇÃO DA AQUISIÇÃO DOS MICRODADOS
# ============================================================
#
# Esta célula percorre todas as URLs configuradas em MICRODADOS_INEP
# e baixa os arquivos ZIP para o diretório data_lake/external.
#
# Cada arquivo é salvo com o mesmo nome presente na URL original.
#
# Entrada:
#
#   MICRODADOS_INEP  Dicionário com ano e URL dos microdados.
#   EXTERNAL_DIR     Diretório local da camada external.
#
# Saída:
#
#   Arquivos ZIP baixados em data_lake/external.
#
# ============================================================
# EXECUÇÃO
# ============================================================

for ano, url in MICRODADOS_INEP.items():
    # Extrai o nome do arquivo a partir do final da URL.
    nome_arquivo = url.split("/")[-1]

    # Monta o caminho completo de destino dentro da camada external.
    destino = EXTERNAL_DIR / nome_arquivo

    print(f"Ano {ano}:")
    baixar_arquivo(url, destino)

In [ ]:
# ============================================================
# CONFIGURAÇÃO DOS ARQUIVOS QUE SERÃO EXTRAÍDOS
# ============================================================
#
# Após o download dos arquivos ZIP, esta célula define quais CSVs
# serão aproveitados pela pipeline.
#
# Nem todos os arquivos presentes nos ZIPs serão usados. Arquivos
# como dicionários, leia-me, inputs e tabelas auxiliares fora do
# escopo são ignorados nesta etapa.
#
# Entrada:
#
#   EXTERNAL_DIR  Diretório da camada external definido nas células
#                 anteriores.
#
# Saída:
#
#   ARQUIVOS_ALVO Lista de CSVs que serão extraídos dos ZIPs.
#   EXTRAIDOS_DIR Diretório onde os CSVs extraídos serão organizados.
#
# ============================================================
# ARQUIVOS-ALVO
# ============================================================

# CSVs necessários para as próximas etapas da pipeline.
ARQUIVOS_ALVO = ["TS_ALUNO.csv", "TS_ESTADO.csv", "TS_MUNICIPIO.csv"]

# ============================================================
# DIRETÓRIO DE EXTRAÇÃO
# ============================================================

# Diretório onde os CSVs extraídos serão salvos, separados por ano.
EXTRAIDOS_DIR = EXTERNAL_DIR / "extraidos"

In [ ]:
# ============================================================
# EXTRAÇÃO DOS CSVS A PARTIR DOS ARQUIVOS ZIP
# ============================================================
#
# Esta célula extrai, de cada ZIP baixado do INEP, apenas os CSVs
# definidos em ARQUIVOS_ALVO.
#
# Os arquivos originais vêm dentro da pasta DADOS/ no ZIP. Durante
# a extração, essa estrutura é simplificada para facilitar o uso
# nas próximas etapas da pipeline.
#
# Exemplo:
#
#   Dentro do ZIP:
#       DADOS/TS_ALUNO.csv
#
#   Após extração:
#       data_lake/external/extraidos/2023/TS_ALUNO.csv
#
# Entrada:
#
#   MICRODADOS_INEP  Dicionário com ano e URL dos arquivos ZIP.
#   EXTERNAL_DIR     Diretório onde os ZIPs foram baixados.
#   EXTRAIDOS_DIR    Diretório de destino dos CSVs extraídos.
#   ARQUIVOS_ALVO    Lista de CSVs usados pela pipeline.
#
# Saída:
#
#   CSVs extraídos e organizados por ano em:
#   data_lake/external/extraidos/{ano}/
#
# ============================================================
# BIBLIOTECAS
# ============================================================

import zipfile

# ============================================================
# FUNÇÃO DE EXTRAÇÃO
# ============================================================

def descompactar(ano: int, zip_path: Path) -> None:
    """
    Extrai os CSVs necessários de um arquivo ZIP do INEP.

    Parâmetros:
        ano: ano de referência dos microdados.
        zip_path: caminho local do arquivo ZIP baixado.

    Retorno:
        Nenhum. Os arquivos são gravados diretamente em disco.
    """

    # Cria a pasta de destino do ano, se ela ainda não existir.
    destino_ano = EXTRAIDOS_DIR / str(ano)
    destino_ano.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as arquivo_zip:
        for alvo in ARQUIVOS_ALVO:
            # Caminho do CSV dentro da estrutura original do ZIP.
            origem_interna = f"DADOS/{alvo}"

            # Caminho final do CSV extraído, sem manter a subpasta DADOS/.
            destino_arquivo = destino_ano / alvo

            # Lê o CSV de dentro do ZIP e grava no destino final.
            with arquivo_zip.open(origem_interna) as fonte, open(destino_arquivo, "wb") as saida:
                saida.write(fonte.read())

            print(f"Extraído: {ano}/{alvo}")

# ============================================================
# EXECUÇÃO
# ============================================================

# Descompacta os arquivos dos anos configurados anteriormente.
for ano, url in MICRODADOS_INEP.items():
    # Monta o caminho local do ZIP a partir do nome presente na URL.
    zip_path = EXTERNAL_DIR / url.split("/")[-1]

    print(f"Ano {ano}:")
    descompactar(ano, zip_path)

print(f"CSVs extraídos em: {EXTRAIDOS_DIR}")

In [ ]:
# ============================================================
# AQUISIÇÃO DAS PLANILHAS DE RESULTADOS E METAS
# ============================================================
#
# Além dos microdados em ZIP, a pipeline também utiliza planilhas
# XLSX com resultados e metas por município e por unidade federativa.
#
# Essas planilhas possuem URLs, caminhos e padrões de nome diferentes
# dos arquivos ZIP de microdados. Por isso, elas são configuradas em
# um dicionário próprio.
#
# Entrada:
#
#   METAS_INEP     Dicionário com identificador e URL das planilhas.
#   EXTRAIDOS_DIR  Diretório onde os arquivos serão organizados por ano.
#
# Saída:
#
#   Arquivos XLSX baixados em:
#   data_lake/external/extraidos/{ano}/
#
# ============================================================
# CONFIGURAÇÃO DAS PLANILHAS
# ============================================================

METAS_INEP = {
    "municipios_2023": "https://download.inep.gov.br/avaliacao_da_alfabetizacao/resultados_e_metas_municipios.xlsx",
    "ufs_2023": "https://download.inep.gov.br/avaliacao_da_alfabetizacao/resultados_e_metas_ufs.xlsx",
    "municipios_2024": "https://download.inep.gov.br/alfabetiza_brasil/resultados_e_metas_municipios_2024.xlsx",
    "ufs_2024": "https://download.inep.gov.br/alfabetiza_brasil/resultados_e_metas_ufs_2024_2.xlsx",
    "municipios_2025": "https://download.inep.gov.br/avaliacao_da_alfabetizacao/resultados/resultados_e_metas_municipios_2025_v2.xlsx",
    "ufs_2025": "https://download.inep.gov.br/avaliacao_da_alfabetizacao/resultados/resultados_e_metas_ufs_2025_v1.xlsx",
}

# ============================================================
# EXECUÇÃO
# ============================================================

for nome, url in METAS_INEP.items():
    # Extrai o ano a partir da chave configurada.
    # Exemplo: municipios_2023 -> 2023
    ano = nome.split("_")[-1]

    # Cria a pasta do ano, caso ela ainda não exista.
    destino_dir = EXTRAIDOS_DIR / ano
    destino_dir.mkdir(parents=True, exist_ok=True)

    # Mantém o nome original do arquivo informado na URL.
    destino = destino_dir / Path(url).name

    print(f"Arquivo: {nome}")
    baixar_arquivo(url, destino)